## 03_esm2_features

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\360Downloads\bioinformatics\Task\AIP


In [3]:
train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
test_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

(3583, 2) (897, 2)


,sequence,label
0,SLLLNGGCKVSNYDE,1
1,DAEFRHDSGYEVHHQ,1
2,GRTGRGKPGIYRFVAPGE,1
3,ASLKPEFVQIINAKN,1
4,KCEFQDAYVLLSEKK,1


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model_name = "facebook/esm2_t12_35M_UR50D"

Device: cpu


In [5]:
# 加载tokenizer和model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model = model.to(device)
model.eval()

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 480, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-11): 12 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=480, out_features=480, bias=True)
            (key): Linear(in_features=480, out_features=480, bias=True)
            (value): Linear(in_features=480, out_features=480, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=480, out_features=480, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((480,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=480, out_features=1920, bias=True)
        )
        (output): EsmOutput(
      

In [6]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = str(seq).upper().strip()
    seq = "".join([aa for aa in seq if aa in VALID_AA])
    return seq

In [7]:
train_df["clean_seq"] = train_df["sequence"].apply(clean_sequence)
test_df["clean_seq"] = test_df["sequence"].apply(clean_sequence)

print("Empty train seq:", (train_df["clean_seq"].str.len() == 0).sum())
print("Empty test seq:", (test_df["clean_seq"].str.len() == 0).sum())

Empty train seq: 0
Empty test seq: 0


In [8]:
# 提取embedding
def extract_esm2_embeddings(sequences, batch_size=16, max_length=1024):
    all_embeddings = []

    for i in tqdm(range(0, len(sequences), batch_size), desc="Extracting ESM2 embeddings"):
        batch_seqs = sequences[i:i + batch_size]

        tokens = tokenizer(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        input_ids = tokens["input_ids"].to(device)
        attention_mask = tokens["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden_dim)

        # mean pooling，忽略 padding
        mask = attention_mask.unsqueeze(-1)  # (batch, seq_len, 1)
        summed = (hidden_states * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / counts

        all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

In [9]:
sample_emb = extract_esm2_embeddings(
    train_df["clean_seq"].tolist()[:10],
    batch_size=4
)

print(sample_emb.shape)
print(sample_emb[:2, :5])

Extracting ESM2 embeddings: 100%|██████████| 3/3 [00:00<00:00,  5.02it/s]

(10, 480)
[[-0.12870646 -0.06898705  0.2150595   0.05817047  0.1280067 ]
 [-0.04763674 -0.00846191  0.23549375  0.0160656  -0.10346796]]


In [69]:
X_train_esm2 = extract_esm2_embeddings(
    train_df["clean_seq"].tolist(),
    batch_size=16
)

X_test_esm2 = extract_esm2_embeddings(
    test_df["clean_seq"].tolist(),
    batch_size=16
)

print("X_train_esm2:", X_train_esm2.shape)
print("X_test_esm2:", X_test_esm2.shape)

Extracting ESM2 embeddings: 100%|██████████| 57/57 [00:14<00:00,  3.88it/s]

X_train_esm2: (3583, 480)
X_test_esm2: (897, 480)


In [70]:
feature_dir = PROJECT_ROOT / "data" / "processed" / "esm2"
feature_dir.mkdir(parents=True, exist_ok=True)

np.save(feature_dir / "X_train_esm2.npy", X_train_esm2)
np.save(feature_dir / "X_test_esm2.npy", X_test_esm2)

y_train = train_df["label"].values
y_test = test_df["label"].values

np.save(feature_dir / "y_train.npy", y_train)
np.save(feature_dir / "y_test.npy", y_test)

print("ESM2 features saved.")

ESM2 features saved.


# esm2_modeling

In [39]:
# 使用上一步提取到的esm2特征重新训练模型，检查预训练特征的效果
from sklearn.preprocessing import StandardScaler

X_train_esm2 = np.load(PROJECT_ROOT / "data" / "processed" / "esm2" / "X_train_esm2.npy")
X_test_esm2 = np.load(PROJECT_ROOT / "data" / "processed" / "esm2" / "X_test_esm2.npy")
y_train = np.load(PROJECT_ROOT / "data" / "processed" / "esm2" / "y_train.npy")
y_test = np.load(PROJECT_ROOT / "data" / "processed" / "esm2" / "y_test.npy")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_esm2)
X_test_scaled = scaler.transform(X_test_esm2)

print(X_train_scaled.shape, X_test_scaled.shape)

(3583, 480) (897, 480)


## (1)LightGBM

In [40]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

from sklearn.metrics import make_scorer, confusion_matrix

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "specificity": make_scorer(specificity_score)
}

def evaluate_model(y_true, y_pred, y_prob):
    metrics = {
        "ACC": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
          
    }
    return metrics


def print_metrics(metrics, title="Metrics"):
    print(f"===== {title} =====")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

In [41]:
# 5-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgb_cv_results = cross_validate(
    lgb_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in lgb_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")

test_accuracy: 0.6503 ± 0.0105
test_precision: 0.5501 ± 0.0152
test_recall: 0.4484 ± 0.0273
test_f1: 0.4938 ± 0.0213
test_roc_auc: 0.6636 ± 0.0159
test_pr_auc: 0.5475 ± 0.0095
test_specificity: 0.7746 ± 0.0111


In [42]:
# 独立测试集评估
lgb_model.fit(X_train_scaled, y_train)

y_pred_lgb = lgb_model.predict(X_test_scaled)
y_prob_lgb = lgb_model.predict_proba(X_test_scaled)[:, 1]

lgb_metrics = evaluate_model(y_test, y_pred_lgb, y_prob_lgb)
print_metrics(lgb_metrics, title="ESM2 + LightGBM Test")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgb, digits=4))

[LightGBM] [Info] Number of positive: 1365, number of negative: 2218
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 122400
[LightGBM] [Info] Number of data points in the train set: 3583, number of used features: 480
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
===== ESM2 + LightGBM Test =====
ACC: 0.6544
Precision: 0.5602
Recall: 0.4357
Specificity: 0.7892
F1: 0.4901
MCC: 0.2391
ROC_AUC: 0.6777
PR_AUC: 0.5780

Confusion Matrix:
[[438 117]
 [193 149]]

Classification Report:
              precision    recall  f1-score   support

           0     0.6941    0.7892    0.7386       555
           1     0.5602    0.4357    0.4901       342

    accuracy                         0.6544       897
   macro avg     0.6271    0.6124    0.6144       897
weighted avg     0

c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## (2)SVM

In [43]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

svm_cv_results = cross_validate(
    svm_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in svm_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")

test_accuracy: 0.6723 ± 0.0094
test_precision: 0.6557 ± 0.0330
test_recall: 0.2967 ± 0.0133
test_f1: 0.4082 ± 0.0149
test_roc_auc: 0.6780 ± 0.0113
test_pr_auc: 0.5700 ± 0.0140
test_specificity: 0.9035 ± 0.0142


In [44]:
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

svm_metrics = evaluate_model(y_test, y_pred_svm, y_prob_svm)
print_metrics(svm_metrics, title="ESM2 + SVM Test")

===== ESM2 + SVM Test =====
ACC: 0.6734
Precision: 0.6433
Recall: 0.3216
Specificity: 0.8901
F1: 0.4288
MCC: 0.2618
ROC_AUC: 0.6919
PR_AUC: 0.5861


## (3)MLP

In [45]:
from sklearn.neural_network import MLPClassifier
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=32,
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)
mlp_cv_results = cross_validate(
    mlp_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in mlp_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")

test_accuracy: 0.6436 ± 0.0126
test_precision: 0.5449 ± 0.0236
test_recall: 0.3941 ± 0.0308
test_f1: 0.4568 ± 0.0243
test_roc_auc: 0.6447 ± 0.0096
test_pr_auc: 0.5246 ± 0.0068
test_specificity: 0.7971 ± 0.0196


In [46]:
mlp_model.fit(X_train_scaled, y_train)

y_pred_mlp = mlp_model.predict(X_test_scaled)
y_prob_mlp = mlp_model.predict_proba(X_test_scaled)[:, 1]

mlp_metrics = evaluate_model(y_test, y_pred_mlp, y_prob_mlp)
print_metrics(mlp_metrics, title="MLP Test")

===== MLP Test =====
ACC: 0.6589
Precision: 0.6034
Recall: 0.3070
Specificity: 0.8757
F1: 0.4070
MCC: 0.2244
ROC_AUC: 0.6540
PR_AUC: 0.5567


## (4)XGBoost

In [47]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

xgb_cv_results = cross_validate(
    xgb_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in xgb_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")

test_accuracy: 0.6578 ± 0.0112
test_precision: 0.5871 ± 0.0281
test_recall: 0.3458 ± 0.0182
test_f1: 0.4349 ± 0.0185
test_roc_auc: 0.6698 ± 0.0150
test_pr_auc: 0.5507 ± 0.0180
test_specificity: 0.8499 ± 0.0162


In [48]:
# 独立测试集评估性能
xgb_model.fit(X_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_metrics = evaluate_model(y_test, y_pred_xgb, y_prob_xgb)
print_metrics(xgb_metrics, title="XGBoost Test")

c:\Users\User\.conda\envs\ysy_env\lib\site-packages\xgboost\training.py:200: UserWarning: [17:49:11] WARNING: C:\Users\task_177465309458303\croot\xgboost-split_1774653200626\work\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


===== XGBoost Test =====
ACC: 0.6656
Precision: 0.6071
Recall: 0.3480
Specificity: 0.8613
F1: 0.4424
MCC: 0.2459
ROC_AUC: 0.6717
PR_AUC: 0.5700


## (5)LR

In [49]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_cv_results = cross_validate(
    lr_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in lr_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")

test_accuracy: 0.6254 ± 0.0224
test_precision: 0.5078 ± 0.0255
test_recall: 0.5795 ± 0.0236
test_f1: 0.5412 ± 0.0237
test_roc_auc: 0.6455 ± 0.0250
test_pr_auc: 0.5186 ± 0.0380
test_specificity: 0.6537 ± 0.0253


In [50]:
# 独立测试集评估性能
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_metrics = evaluate_model(y_test, y_pred_lr, y_prob_lr)
print_metrics(lr_metrics, title="Logistic Regression Test")

===== Logistic Regression Test =====
ACC: 0.6042
Precision: 0.4837
Recall: 0.5643
Specificity: 0.6288
F1: 0.5209
MCC: 0.1888
ROC_AUC: 0.6282
PR_AUC: 0.5178


## (6)RF

In [51]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_cv_results = cross_validate(
    rf_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in rf_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")


test_accuracy: 0.6458 ± 0.0043
test_precision: 0.6200 ± 0.0228
test_recall: 0.1832 ± 0.0147
test_f1: 0.2823 ± 0.0170
test_roc_auc: 0.6654 ± 0.0076
test_pr_auc: 0.5407 ± 0.0079
test_specificity: 0.9306 ± 0.0098


In [52]:
# 独立测试集评估性能
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_metrics = evaluate_model(y_test, y_pred_rf, y_prob_rf)
print_metrics(rf_metrics, title="Random Forest Test")

===== Random Forest Test =====
ACC: 0.6455
Precision: 0.6364
Recall: 0.1637
Specificity: 0.9423
F1: 0.2605
MCC: 0.1732
ROC_AUC: 0.6844
PR_AUC: 0.5661


## (7)ET

In [53]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

et_cv_results = cross_validate(
    et_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

for metric_name, values in et_cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f} ± {values.std():.4f}")


test_accuracy: 0.6478 ± 0.0066
test_precision: 0.6389 ± 0.0318
test_recall: 0.1736 ± 0.0103
test_f1: 0.2730 ± 0.0151
test_roc_auc: 0.6696 ± 0.0108
test_pr_auc: 0.5435 ± 0.0138
test_specificity: 0.9396 ± 0.0058


In [54]:
# 独立测试集评估性能
et_model.fit(X_train_scaled, y_train)

y_pred_et = et_model.predict(X_test_scaled)
y_prob_et = et_model.predict_proba(X_test_scaled)[:, 1]

et_metrics = evaluate_model(y_test, y_pred_et, y_prob_et)
print_metrics(et_metrics, title="Extra Trees Test")

===== Extra Trees Test =====
ACC: 0.6410
Precision: 0.6316
Recall: 0.1404
Specificity: 0.9495
F1: 0.2297
MCC: 0.1568
ROC_AUC: 0.6736
PR_AUC: 0.5440


# esm特征结果评估

## (1)table

In [55]:
results_df_esm2 = pd.DataFrame([
    {"Model": "ESM2 + LightGBM", **lgb_metrics},
    {"Model": "ESM2 + SVM", **svm_metrics},
    {"Model": "ESM2 + MLP", **mlp_metrics},
    {"Model": "ESM2 + XGBoost", **xgb_metrics},
    {"Model": "ESM2 + Logistic Regression", **lr_metrics},
    {"Model": "ESM2 + Random Forest", **rf_metrics},
    {"Model": "ESM2 + Extra Trees", **et_metrics},
])

results_df_esm2

,Model,ACC,Precision,Recall,Specificity,F1,MCC,ROC_AUC,PR_AUC
0,ESM2 + LightGBM,0.654404,0.560150,0.435673,0.789189,0.490132,0.239122,0.677736,0.578017
1,ESM2 + SVM,0.673356,0.643275,0.321637,0.890090,0.428850,0.261801,0.691889,0.586141
2,ESM2 + MLP,0.658863,0.603448,0.307018,0.875676,0.406977,0.224408,0.654028,0.556709
3,ESM2 + XGBoost,0.665552,0.607143,0.347953,0.861261,0.442379,0.245903,0.671651,0.569958
4,ESM2 + Logistic Regression,0.604236,0.483709,0.564327,0.628829,0.520918,0.188785,0.628197,0.517811
5,ESM2 + Random Forest,0.645485,0.636364,0.163743,0.942342,0.260465,0.173220,0.684432,0.566053
6,ESM2 + Extra Trees,0.641026,0.631579,0.140351,0.949550,0.229665,0.156799,0.673587,0.543990


In [56]:
result_dir = PROJECT_ROOT / "results" / "tables"
result_dir.mkdir(parents=True, exist_ok=True)

results_df_esm2.to_csv(result_dir / "esm2_model_results_with_torch_mlp.csv", index=False)
print("Saved.")

Saved.


In [57]:
def summarize_cv_results(cv_results, model_name):
    summary = {
        "Model": model_name,
        "ACC": np.mean(cv_results["test_accuracy"]),
        "Precision": np.mean(cv_results["test_precision"]),
        "Recall": np.mean(cv_results["test_recall"]),
        "Specificity":np.mean(cv_results["test_specificity"]),
        "F1": np.mean(cv_results["test_f1"]),
        "ROC_AUC": np.mean(cv_results["test_roc_auc"]),
        "PR_AUC": np.mean(cv_results["test_pr_auc"])
    }
    return summary

In [58]:
cv_summary_list = []

cv_summary_list.append(summarize_cv_results(lgb_cv_results, "LightGBM"))
cv_summary_list.append(summarize_cv_results(svm_cv_results, "SVM"))
cv_summary_list.append(summarize_cv_results(xgb_cv_results, "XGBoost"))
cv_summary_list.append(summarize_cv_results(mlp_cv_results, "MLP"))
cv_summary_list.append(summarize_cv_results(lr_cv_results, "LogReg"))
cv_summary_list.append(summarize_cv_results(rf_cv_results, "RandomForest"))
cv_summary_list.append(summarize_cv_results(et_cv_results, "ExtraTrees"))

cv_df = pd.DataFrame(cv_summary_list)
cv_df

,Model,ACC,Precision,Recall,Specificity,F1,ROC_AUC,PR_AUC
0,LightGBM,0.650289,0.550142,0.448352,0.774566,0.493804,0.663600,0.547517
1,SVM,0.672338,0.655686,0.296703,0.903514,0.408207,0.678048,0.569983
2,XGBoost,0.657830,0.587133,0.345788,0.849865,0.434907,0.669759,0.550744
3,MLP,0.643589,0.544933,0.394139,0.797103,0.456804,0.644705,0.524648
4,LogReg,0.625450,0.507814,0.579487,0.653735,0.541181,0.645492,0.518608
5,RandomForest,0.645828,0.619981,0.183150,0.930571,0.282329,0.665420,0.540701
6,ExtraTrees,0.647779,0.638854,0.173626,0.939585,0.272986,0.669573,0.543491
